# Sales Prediction with Advertising Data

This notebook uses `Advertising.csv` to:
- clean and transform data,
- train regression models to forecast sales,
- analyze how advertising changes affect sales,
- produce actionable marketing insights.

> Note: The dataset does not include explicit `target segment` or `platform` fields. We create **proxy features**:
> - `target_segment`: low/medium/high based on total ad spend,
> - `platform`: dominant spend channel among TV/Radio/Newspaper.

In [2]:
# -----------------------------
# 1) Imports and data loading
# -----------------------------
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv('Advertising.csv')

# -----------------------------
# 2) Data cleaning
# -----------------------------
# Drop unnamed index column if present
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])
if "" in df.columns:
    df = df.drop(columns=[""])

# Standardize column names for easier handling
# e.g., TV -> tv, Sales -> sales
df.columns = [col.strip().lower() for col in df.columns]

# Ensure numeric columns are numeric
numeric_base_cols = ["tv", "radio", "newspaper", "sales"]
for col in numeric_base_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove duplicate rows and reset index
df = df.drop_duplicates().reset_index(drop=True)

# -----------------------------
# 3) Feature engineering
# -----------------------------
# Total spend gives a compact view of budget intensity
df["total_spend"] = df["tv"] + df["radio"] + df["newspaper"]

# "target_segment" proxy: classify campaigns by total ad spend bands
# - low: lower budget
# - medium: average budget
# - high: higher budget
df["target_segment"] = pd.qcut(
    df["total_spend"],
    q=3,
    labels=["low", "medium", "high"],
)

# "platform" proxy: dominant spend channel for each record
channel_cols = ["tv", "radio", "newspaper"]
df["platform"] = df[channel_cols].idxmax(axis=1)

# Final feature set for prediction
feature_cols = ["tv", "radio", "newspaper", "total_spend", "target_segment", "platform"]
target_col = "sales"

X = df[feature_cols]
y = df[target_col]

# -----------------------------
# 4) Train/test split
# -----------------------------
# Keep 20% unseen for evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

# -----------------------------
# 5) Preprocessing + model
# -----------------------------
# Numeric features: impute missing values + scale
numeric_features = ["tv", "radio", "newspaper", "total_spend"]
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

# Categorical features: impute missing + one-hot encode
categorical_features = ["target_segment", "platform"]
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

# Combine preprocessing for mixed data types
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# Linear Regression gives interpretable impact estimates
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression()),
    ]
)

# Train model
model.fit(X_train, y_train)

# -----------------------------
# 6) Evaluation metrics
# -----------------------------
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("=== Sales Forecasting Model Performance ===")
print(f"Rows used: {len(df)}")
print(f"Train rows: {len(X_train)} | Test rows: {len(X_test)}")
print(f"MAE  : {mae:.3f}")
print(f"RMSE : {rmse:.3f}")
print(f"R²   : {r2:.3f}")

# Save reusable objects for next analysis cells
trained_df = df.copy()
trained_model = model
print("\nModel and cleaned data saved as: trained_model, trained_df")

=== Sales Forecasting Model Performance ===
Rows used: 200
Train rows: 160 | Test rows: 40
MAE  : 1.036
RMSE : 1.356
R²   : 0.942

Model and cleaned data saved as: trained_model, trained_df


In [3]:
# ---------------------------------------------
# 7) Impact analysis: how ad spend changes sales
# ---------------------------------------------
# Build scenarios by changing one channel at a time from the test set baseline.
# This approximates "what-if" impact for planning.

baseline = X_test.copy()

# Helper function to apply % change to one channel and recompute derived fields
def scenario_with_channel_change(input_df, channel, pct_change):
    scenario = input_df.copy()

    # Apply percentage change, e.g., +10% => *1.10
    scenario[channel] = scenario[channel] * (1 + pct_change)

    # Recompute engineered features to keep transformations consistent
    scenario["total_spend"] = scenario["tv"] + scenario["radio"] + scenario["newspaper"]
    scenario["target_segment"] = pd.qcut(
        scenario["total_spend"].rank(method="first"),
        q=3,
        labels=["low", "medium", "high"],
    )
    scenario["platform"] = scenario[["tv", "radio", "newspaper"]].idxmax(axis=1)

    return scenario

# Baseline predicted sales
baseline_pred = trained_model.predict(baseline).mean()

# Evaluate incremental impact for +10% and -10% per channel
impact_rows = []
for channel in ["tv", "radio", "newspaper"]:
    for pct in [-0.10, 0.10]:
        scen = scenario_with_channel_change(baseline, channel, pct)
        scen_pred = trained_model.predict(scen).mean()

        impact_rows.append(
            {
                "channel": channel,
                "change_pct": pct,
                "predicted_avg_sales": round(float(scen_pred), 3),
                "delta_vs_baseline": round(float(scen_pred - baseline_pred), 3),
            }
        )

impact_df = pd.DataFrame(impact_rows).sort_values(["channel", "change_pct"])

print("=== Advertising Impact (What-if on Test Set) ===")
print(f"Baseline avg predicted sales: {baseline_pred:.3f}\n")
print(impact_df.to_string(index=False))

# ---------------------------------------------
# 8) Simple future forecast scenarios
# ---------------------------------------------
# Create a few planning scenarios for next campaign cycles.
future_scenarios = pd.DataFrame(
    [
        {"scenario": "Conservative", "tv": 120, "radio": 20, "newspaper": 15},
        {"scenario": "Balanced Growth", "tv": 180, "radio": 30, "newspaper": 20},
        {"scenario": "Aggressive Reach", "tv": 240, "radio": 40, "newspaper": 25},
    ]
)

# Rebuild engineered proxy features for scenario scoring
future_scenarios["total_spend"] = (
    future_scenarios["tv"] + future_scenarios["radio"] + future_scenarios["newspaper"]
)
future_scenarios["target_segment"] = pd.qcut(
    future_scenarios["total_spend"].rank(method="first"),
    q=3,
    labels=["low", "medium", "high"],
)
future_scenarios["platform"] = future_scenarios[["tv", "radio", "newspaper"]].idxmax(axis=1)

# Predict sales for each scenario
future_scenarios["predicted_sales"] = trained_model.predict(
    future_scenarios[["tv", "radio", "newspaper", "total_spend", "target_segment", "platform"]]
)

print("\n=== Future Sales Forecast Scenarios ===")
print(
    future_scenarios[
        ["scenario", "tv", "radio", "newspaper", "target_segment", "platform", "predicted_sales"]
    ].to_string(index=False)
)

# ---------------------------------------------
# 9) Actionable business insights
# ---------------------------------------------
print("\n=== Actionable Marketing Insights ===")
print("1) Prioritize channels showing larger positive delta in the impact table.")
print("2) Use the Balanced Growth scenario as a baseline and adjust toward the best-performing channel.")
print("3) If budget is constrained, reduce spend from channels with smaller sensitivity first.")
print("4) Re-train monthly/quarterly with fresh campaign data to keep forecasts accurate.")

=== Advertising Impact (What-if on Test Set) ===
Baseline avg predicted sales: 13.807

  channel  change_pct  predicted_avg_sales  delta_vs_baseline
newspaper        -0.1               13.764             -0.043
newspaper         0.1               13.774             -0.033
    radio        -0.1               13.373             -0.433
    radio         0.1               14.165              0.358
       tv        -0.1               13.110             -0.697
       tv         0.1               14.428              0.621

=== Future Sales Forecast Scenarios ===
        scenario  tv  radio  newspaper target_segment platform  predicted_sales
    Conservative 120     20         15            low       tv        13.246582
 Balanced Growth 180     30         20         medium       tv        16.969554
Aggressive Reach 240     40         25           high       tv        21.369258

=== Actionable Marketing Insights ===
1) Prioritize channels showing larger positive delta in the impact table.
2) Us